In [1]:
import pprint
import urllib.parse
import json5
from qwen_agent.agents import Assistant
from qwen_agent.tools.base import BaseTool, register_tool
import os

In [2]:
# 步骤 1（可选）：添加一个名为 `my_image_gen` 的自定义工具。
@register_tool('my_image_gen')
class MyImageGen(BaseTool):
    # `description` 用于告诉智能体该工具的功能。
    description = 'AI 绘画（图像生成）服务，输入文本描述，返回基于文本信息绘制的图像 URL。'
    # `parameters` 告诉智能体该工具有哪些输入参数。
    parameters = [{
        'name': 'prompt',
        'type': 'string',
        'description': '期望的图像内容的详细描述',
        'required': True
    }]

    def call(self, params: str, **kwargs) -> str:
        # `params` 是由 LLM 智能体生成的参数。
        prompt = json5.loads(params)['prompt']
        prompt = urllib.parse.quote(prompt)
        return json5.dumps(
            {'image_url': f'https://image.pollinations.ai/prompt/{prompt}'},
            ensure_ascii=False)

In [3]:
from dotenv import load_dotenv
load_dotenv()
# 步骤 2：配置您所使用的 LLM。
llm_cfg = {
    # 使用 DashScope 提供的模型服务：
    'model': 'qwen-max',
    'model_server': 'dashscope',
    'api_key': os.getenv('DASHSCOPE_API_KEY'),  # 从环境变量获取API Key
    'generate_cfg': {
        'top_p': 0.8
    }
}

llm_cfg = {
    # 使用 DashScope 提供的模型服务：
    'model': 'deepseek-v3',
    'model_server': 'https://dashscope.aliyuncs.com/compatible-mode/v1',
    'api_key': os.getenv('DASHSCOPE_API_KEY'),  # 从环境变量获取API Key
    'generate_cfg': {
        'top_p': 0.8
    }
}

In [4]:
# 步骤 3：创建一个智能体。这里我们以 `Assistant` 智能体为例，它能够使用工具并读取文件。
system_instruction = '''你是一个乐于助人的AI助手。
在收到用户的请求后，你应该：
- 首先绘制一幅图像，得到图像的url，
- 然后运行代码`request.get`以下载该图像的url，
- 最后从给定的文档中选择一个图像操作进行图像处理。
用 `plt.show()` 展示图像。
你总是用中文回复用户。'''
tools = ['my_image_gen', 'code_interpreter']  # `code_interpreter` 是框架自带的工具，用于执行代码。
import os
# 获取文件夹下所有文件
file_dir = os.path.join('./', 'docs')
files = []
if os.path.exists(file_dir):
    # 遍历目录下的所有文件
    for file in os.listdir(file_dir):
        file_path = os.path.join(file_dir, file)
        if os.path.isfile(file_path):  # 确保是文件而不是目录
            files.append(file_path)
print('files=', files)

files= ['./docs\\1-平安商业综合责任保险（亚马逊）.txt', './docs\\2-雇主责任险.txt', './docs\\3-平安企业团体综合意外险.txt', './docs\\4-雇主安心保.txt', './docs\\5-施工保.txt', './docs\\6-财产一切险.txt', './docs\\7-平安装修保.txt', './docs\\平安产险交通出行意外伤害保险（互联网版）产品说明.pdf', './docs\\平安产险交通工具意外伤害保险（互联网版）条款.pdf', './docs\\平安企业团体综合意外险(互联网版)适用条款.pdf', './docs\\平安商业综合责任保险（亚马逊）.pdf', './docs\\平安境内紧急医疗救援服务条款.pdf', './docs\\平安附加疾病身故保险条款.pdf', './docs\\浦发上海浦东发展银行西安分行个金客户经理考核办法.pdf']


In [5]:
bot = Assistant(llm=llm_cfg,
                system_message=system_instruction,
                function_list=tools,
                files=files)

In [7]:
# 步骤 4：作为聊天机器人运行智能体。
messages = []  # 这里储存聊天历史。
query = "介绍下雇主责任险"
# 将用户请求添加到聊天历史。
messages.append({'role': 'user', 'content': query})
response = []
current_index = 0
for response in bot.run(messages=messages):
    if current_index == 0:
        # 尝试获取并打印召回的文档内容
        if hasattr(bot, 'retriever') and bot.retriever:
            print("\n===== 召回的文档内容 =====")
            retrieved_docs = bot.retriever.retrieve(query)
            if retrieved_docs:
                for i, doc in enumerate(retrieved_docs):
                    print(f"\n文档片段 {i+1}:")
                    print(f"内容: {doc.page_content}")
                    print(f"元数据: {doc.metadata}")
            else:
                print("没有召回任何文档内容")
            print("===========================\n")

    current_response = response[0]['content'][current_index:]
    current_index = len(response[0]['content'])
    print(current_response, end='')

2026-04-25 23:52:30,118 - memory.py - 169 - INFO - {"keywords_zh": ["雇主", "责任险", "保险", "雇主责任", "责任保险"], "keywords_en": ["employer", "liability", "insurance", "employer liability", "liability insurance"], "text": "介绍下雇主责任险"}
2026-04-25 23:52:32,745 - simple_doc_parser.py - 427 - INFO - Start parsing ./docs\1-平安商业综合责任保险（亚马逊）.txt...
2026-04-25 23:52:32,761 - simple_doc_parser.py - 475 - INFO - Finished parsing ./docs\1-平安商业综合责任保险（亚马逊）.txt. Time spent: 0.016605377197265625 seconds.
2026-04-25 23:52:32,774 - doc_parser.py - 122 - INFO - Start chunking ./docs\1-平安商业综合责任保险（亚马逊）.txt (1-平安商业综合责任保险（亚马逊）.txt)...
2026-04-25 23:52:32,776 - doc_parser.py - 140 - INFO - Finished chunking ./docs\1-平安商业综合责任保险（亚马逊）.txt (1-平安商业综合责任保险（亚马逊）.txt). Time spent: 0.0 seconds.
2026-04-25 23:52:32,786 - simple_doc_parser.py - 427 - INFO - Start parsing ./docs\2-雇主责任险.txt...
2026-04-25 23:52:32,799 - simple_doc_parser.py - 475 - INFO - Finished parsing ./docs\2-雇主责任险.txt. Time spent: 0.013508319854736328 seconds.



================ 拼接后的完整prompt（system prompt部分） ================
你是一个乐于助人的AI助手。
在收到用户的请求后，你应该：
- 首先绘制一幅图像，得到图像的url，
- 然后运行代码`request.get`以下载该图像的url，
- 最后从给定的文档中选择一个图像操作进行图像处理。
用 `plt.show()` 展示图像。
你总是用中文回复用户。

# 知识库

## 来自 [文件](1-平安商业综合责任保险（亚马逊）.txt) 的内容：

```
【平安商业综合责任保险（亚马逊）】

Q1：为什么一定要投保？
1. 平台要求：
从2021年9月1日起，亚马逊要求商城任意一个月总销售额超过1万美元的卖家，必须在30天内投保“商业综合责任保险”（Commercial General Liability），否则可能无法提现甚至禁止销售。
2.	风险隐患：
北美的法律有集体诉讼、高额诉讼、惩罚性赔偿、消费者导向等特点，产品缺陷可能引发巨额赔偿。

Q2：哪些卖家需要投保？
2021年9月1日后，在亚马逊商城任意一个月总销售收入超过10000美元，或在亚马逊要求的其他情况下，必须在30天内投保商业责任保险，并将亚马逊指定为附加被保险人。

Q3:产品亮点（平安商业综合责任保险（亚马逊））
1.	产品大类覆盖广
我司方案覆盖产品大类多达195个，超2000种商品，覆盖亚马逊网站95%以上在售产品，满足大部分卖家保险需求。
2.	保障范围全
涵盖法律费用、产品和完工责任，完全符合亚马逊要求。
3.	服务响应快
承保成功后即可获取英文保单及保险凭证，便于卖家快速通过平台审核。
4.	续保更优惠
我司方案为上年在我司投保商业综合保险的优质客户提供续保优惠套餐。

Q4. 保障方案
保障内容	产品与完工责任
责任限额	每次及累计赔偿限额USD100万
保单触发机制	事故发生制
免赔	无免赔/10000美金免赔两种方案可选
承保范围	全球（含美加）
保费收取	按销售额分档收费，最低184美元

Q5 理赔流程
1. 三者报案
在事故发生后，海外三者会将事故报给亚马逊平台并发起索赔。
2.	公估处理
亚马逊委托公估公司代表其对接处理案件，公估将初步评估海外三者损失，并通过邮件告知平台商家（被保险人），部分案件将通过代理同

In [8]:
# 将机器人的回应添加到聊天历史
messages.extend(response)
print(messages)

[{'role': 'user', 'content': '介绍下雇主责任险'}, {'role': 'assistant', 'content': '雇主责任险是一种商业保险，旨在帮助企业转移因员工在工作期间发生意外伤害或职业病而产生的经济赔偿责任。以下是雇主责任险的主要内容和特点：\n\n### 1. **保障范围**\n- **死亡赔偿金**：员工因工作原因导致身故，保险公司按约定金额赔付。\n- **伤残赔偿金**：员工因工作原因导致伤残，根据伤残等级按比例赔付。\n- **医疗费用**：涵盖员工因工伤或职业病产生的合理医疗费用。\n- **误工费用**：补偿员工因工伤无法工作期间的工资损失。\n- **法律诉讼费**：企业因工伤纠纷产生的法律费用。\n\n### 2. **适用对象**\n- 任何有用工需求的单位，包括企业、个体工商户等。\n- 覆盖正式员工、临时工、实习生等（需符合劳动合同或事实劳动关系）。\n\n### 3. **产品亮点**\n- **风险转移**：将企业对员工的赔偿责任转嫁给保险公司。\n- **法律合规**：符合《工伤保险条例》等法律法规要求。\n- **灵活定制**：可根据企业需求选择不同保障方案（如A/B/94款）。\n- **税前列支**：保费可计入企业成本，减少所得税负担。\n\n### 4. **与工伤保险的区别**\n- **补充作用**：工伤保险无法覆盖的部分（如一次性伤残就业补助金、误工费）可由雇主责任险补充。\n- **快速理赔**：相比工伤保险的复杂流程，雇主责任险理赔更高效。\n\n### 5. **理赔案例**\n- **案例1**：员工上班途中车祸身亡，获赔30万元。\n- **案例2**：工人高空作业坠亡，赔付95万元。\n\n### 6. **投保建议**\n- **高风险行业**（如建筑、制造）应优先投保。\n- **结合工伤保险**：双重保障更全面。\n\n如果需要更详细的产品方案或具体案例，可以进一步咨询！'}]
